<a href="https://colab.research.google.com/github/christianadeyemi/Cheminformatics/blob/main/Amide_Rxn_Enumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**
This Notebook is gives a step by step tutorial in performing reaction-based enumeration of amide coupling of carboxylic acids and a single primary amine.

Install necessary python packages

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pandas rdkit mols2grid

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import mols2grid
from itertools import product

In [ ]:
reaction_smarts = "[C:1][C:2](=[O:3])[OH].[n:21]1[c:5]([NH2:4])[c:6][c:7]([NH:8][C:9](=[O:10])[c:11]2[c:12]([Cl:13])[c:14][c:15][c:16][c:17]2[Cl:18])[c:19][c:20]1>>[C:1][C:2](=[O:3])[NH:4][c:5]1[n:21]cc([NH:8][C:9](=[O:10])[c:11]2[c:12]([Cl:13])[c:14][c:15][c:16][c:17]2[Cl:18])cc1"

In [ ]:
reaction_mol = AllChem.ReactionFromSmarts(reaction_smarts)
reaction_mol

In [ ]:
acid_mol = Chem.MolFromSmiles("CC(=O)O")
acid_mol


In [ ]:
amine_mol = Chem.MolFromSmiles("Nc2cc(NC(=O)c1c(Cl)cccc1Cl)ccn2")
amine_mol

In [ ]:

df_acids = pd.read_csv("/content/drive/MyDrive/data/Enamine_Carboxylic_Acids.csv")
df_acids

In [ ]:
df_acids = df_acids.drop(['IUPAC Name','URL','Stock_weight_G','Subclass'], axis=1, errors='ignore')
df_acids

In [ ]:
df_acids['mol'] = df_acids.SMILES.apply(Chem.MolFromSmiles)
df_acids

In [ ]:
patt = Chem.MolFromSmarts("CC(=O)O")
df_acids["matches_smarts"] = df_acids["mol"].apply(
    lambda m: m.HasSubstructMatch(patt)
)
df_acids_filtered = df_acids[df_acids["matches_smarts"]].copy()
df_acids_filtered

In [ ]:
mols2grid.display(df_acids_filtered,subset=["img","Catalog_ID"])

In [ ]:
df1_acids = df_acids_filtered[df_acids_filtered['Mw'] < 200]
df1_acids

In [ ]:
data = {
   'SMILES': ['Nc2cc(NC(=O)c1c(Cl)cccc1Cl)ccn2'],
   'class': ['amine']
}
df_amine = pd.DataFrame(data)
print(df_amine)

In [ ]:
df_amine['mol'] = df_amine.SMILES.apply(Chem.MolFromSmiles)
df_amine

In [ ]:
acid_list  = list(zip(df1_acids['mol'], df1_acids['Catalog_ID']))
amine_list = df_amine['mol'].tolist()

In [ ]:
print("Amine list length:", len(amine_list))
print("Acid list length:", len(acid_list))

In [ ]:

def enumerate_acid_library(reaction_mol, amine_mol, acid_list):
    products = []

    for acid_mol, acid_name in acid_list:
        if acid_mol is None:
            continue

        prod = reaction_mol.RunReactants((acid_mol, amine_mol))

        if prod and len(prod):
            try:
                product_mol = prod[0][0]
                Chem.SanitizeMol(product_mol)
                smi = Chem.MolToSmiles(product_mol, canonical=True)
                products.append([smi, acid_name])
            except:
                continue

    return products


In [ ]:
products = enumerate_acid_library(reaction_mol, amine_list[0], acid_list)

In [ ]:
prod_df = pd.DataFrame(products, columns=["SMILES", "Acid_Catalog_ID"])
prod_df.head()

In [ ]:
mols2grid.display(prod_df,subset=["img","Acid_Catalog_ID"])

In [ ]:
prod_df.to_csv
prod_df

In [ ]:
prod_df['mol'] = prod_df.SMILES.apply(Chem.MolFromSmiles)
prod_df

**Calculate Lipinski descriptor**

In [ ]:
from rdkit.Chem.Descriptors import MolWt, MolLogP, NumAromaticRings, NumHDonors, NumHAcceptors


In [ ]:
def calc_descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mw, logp, num_arom_rings, hbd, hba = [x(mol) for x in [MolWt, MolLogP, NumAromaticRings, NumHDonors, NumHAcceptors]]
        result = [mw, num_arom_rings, logp, hbd, hba]
    else:
        result = [None] * 5
    return result

In [ ]:
prod_df['desc'] = prod_df.SMILES.apply(calc_descriptors)

In [ ]:
desc_columns = ['MW','NumAromatic', 'LogP','HBD','HBA']
prod_df[desc_columns] = prod_df.desc.to_list()

In [ ]:
prod_df.drop("desc",axis=1,inplace=True, errors='ignore')

In [ ]:
prod_df